# Qdrant Learning Lab

Run each cell with **Shift+Enter**. Every section has a **→ Qdrant UI** note
pointing you to the exact place in https://cloud.qdrant.io to confirm what
the code shows visually.

**Prerequisites**
- `.env` at repo root with `QDRANT_URL` and `QDRANT_API_KEY`
- Run `task setup` once to install Python deps
- Ollama running locally for the embedding sections (`ollama serve`)

**Sections**
1. Connect and verify
2. Collections — what they are and what they store
3. Points and payloads — browsing raw data
4. HNSW — how the index works, what the parameters mean
5. Payload indexes and filtering
6. Multi-tenancy via payload filtering
7. Quantization — memory vs recall
8. Real semantic query against your corpus

---
## Setup — run this first

In [ ]:
import os, random, time, sys
from pathlib import Path
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, HnswConfigDiff,
    PointStruct, PayloadSchemaType,
    FieldCondition, Filter, MatchValue,
    SearchParams,
    ScalarQuantization, ScalarQuantizationConfig, ScalarType,
)

# Load .env from repo root (two levels up from notebooks/)
REPO_ROOT = Path().resolve().parent
load_dotenv(REPO_ROOT / ".env")

QDRANT_URL     = os.environ["QDRANT_URL"]
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY") or None
COLLECTION     = os.environ.get("COLLECTION", "platform-docs")

qd = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=30, check_compatibility=False)

print("Connected to:", QDRANT_URL)
print("Collections:", [c.name for c in qd.get_collections().collections])

---
## Part 1 — Collections

A **collection** is Qdrant's equivalent of a table — but instead of rows it
stores **points**. Every point in a collection must share the same vector
dimension and distance metric. You set these at creation time and **cannot
change them without re-indexing**.

Each point contains:
- `id` — unique integer or UUID
- `vector` — list of floats (the embedding)
- `payload` — arbitrary JSON (source file, tenant, doc_type, etc.)

**→ Qdrant UI:** https://cloud.qdrant.io → your cluster → **Collections** tab.  
You'll see each collection listed with its point count. The fields printed
below (vector size, distance, HNSW config) are in the **Info** tab of each
collection. Click your collection name, then the Info tab.

In [ ]:
# ── Inspect the platform-docs collection ────────────────────────────────
if not qd.collection_exists(COLLECTION):
    print(f"Collection '{COLLECTION}' not found.")
    print("Run: task w0:migrate   (this creates the collection and ingests docs)")
else:
    info = qd.get_collection(COLLECTION)
    cfg  = info.config.params
    hnsw = cfg.hnsw_config

    print(f"Collection : {COLLECTION}")
    print(f"  Points total      : {info.points_count:,}")
    print(f"  Indexed vectors   : {info.indexed_vectors_count:,}")
    print(f"    ^ if this < points_count, HNSW is still building — normal after ingest")
    print(f"  Segments          : {info.segments_count}")
    print(f"    ^ grows during ingest, shrinks as optimizer merges them")
    print(f"  Optimizer status  : {info.optimizer_status.status}")
    print()
    print(f"Vector config")
    print(f"  size     : {cfg.vectors.size}  (must match embedding model output)")
    print(f"  distance : {cfg.vectors.distance}  (set at creation, cannot change)")
    print()
    print(f"HNSW config")
    print(f"  m              : {hnsw.m}  (connections per node in graph)")
    print(f"  ef_construct   : {hnsw.ef_construct}  (search width during index build)")
    print(f"  full_scan_threshold: {hnsw.full_scan_threshold}")
    print(f"    ^ collections smaller than this skip HNSW and use brute force")
    print()
    print(f"Indexed payload fields (these are fast to filter on):")
    for field, schema in (info.payload_schema or {}).items():
        print(f"  {field:20s} : {schema.data_type}")
    if not info.payload_schema:
        print("  (none — run task w0:migrate to create tenant + doc_type indexes)")

---
## Part 2 — Points and Payloads

`scroll()` browses raw points without running a similarity search. Think of it
as `SELECT * FROM collection WHERE ...` — no embedding needed.

Use `scroll()` to:
- Verify payloads are correct after ingest (are `tenant` and `doc_type` set?)
- Spot-check chunk quality (coherent text or mid-sentence splits?)
- Paginate through large collections without a query

**→ Qdrant UI:** Collection → **Points** tab.  
You see a paginated table of all points with their payloads. Click any point
ID to expand its full JSON payload and the raw vector values. The `scroll()`
call below returns identical data via the API.

In [ ]:
# Browse 5 points — no query vector required
points, next_offset = qd.scroll(
    collection_name=COLLECTION,
    limit=5,
    with_payload=True,
    with_vectors=False,   # change to True to see the float array
)

print(f"Returned {len(points)} points  (next_offset={next_offset})\n")
for p in points:
    print(f"ID: {p.id}")
    for k, v in p.payload.items():
        display = (str(v)[:100] + "...") if len(str(v)) > 100 else str(v)
        print(f"  {k:12s}: {display}")
    print()

In [ ]:
# Filtered scroll — only show points with a specific doc_type
# Change the value below to explore different doc types
points_filtered, _ = qd.scroll(
    collection_name=COLLECTION,
    scroll_filter=Filter(must=[
        FieldCondition(key="doc_type", match=MatchValue(value="architecture"))
    ]),
    limit=3,
    with_payload=True,
    with_vectors=False,
)

print(f"Points with doc_type=architecture: {len(points_filtered)}")
for p in points_filtered:
    print(f"  id={p.id}  source={p.payload.get('source','?')}")
    print(f"  text: {p.payload.get('text','')[:120]}...")
    print()

---
## Part 3 — HNSW: How the Index Works

HNSW (Hierarchical Navigable Small World) is a **layered graph**:

```
Layer 2  •————•                        ← few 'highway' nodes, long-range links
              |
Layer 1  •——•——•——•——•                  ← more nodes, medium-range links
              |
Layer 0  •—•—•—•—•—•—•—•—•—•           ← ALL points, final result set
                    ↑ query starts at top, descends to Layer 0
```

A query enters the top layer, walks greedily toward the nearest node, descends
to the next layer at that node, walks again, repeats until Layer 0. This is
O(log N) instead of O(N) for brute force.

| Parameter | Set at | Effect |
|-----------|--------|---------|
| `m` | collection creation | Connections per node. Higher → better recall, more RAM |
| `ef_construct` | collection creation | Search width during build. Higher → better graph, slower ingest |
| `hnsw_ef` | query time | Search width at query. Higher → better recall, more latency. **No rebuild needed.** |
| `exact=True` | query time | Bypass HNSW entirely, brute-force. Use for ground truth. |

**→ Qdrant UI:** Collection → **Info** tab → HNSW Configuration section.  
You'll see `m` and `ef_construct` here. These cannot be changed in the UI —
they require re-creating the collection. The next cell creates a scratch
collection with non-default values so you can verify the UI reflects them.

In [ ]:
SCRATCH = "notebook-scratch"

# Clean up from any previous run
if qd.collection_exists(SCRATCH):
    qd.delete_collection(SCRATCH)

# Create with high-recall HNSW (m=32, ef_construct=200 vs defaults of 16, 100)
qd.create_collection(
    collection_name=SCRATCH,
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
    hnsw_config=HnswConfigDiff(m=32, ef_construct=200),
)
print(f"Created '{SCRATCH}'")

# Verify via API — UI will show the same values
info = qd.get_collection(SCRATCH)
hnsw = info.config.params.hnsw_config
print(f"  m            = {hnsw.m}           (default is 16)")
print(f"  ef_construct = {hnsw.ef_construct}         (default is 100)")
print()
print("→ Qdrant UI: refresh Collections list — 'notebook-scratch' appears.")
print("  Click it → Info tab → HNSW Configuration → confirm m=32, ef_construct=200")

In [ ]:
# Upsert 50 random 4-dimensional points
random.seed(42)
points = [
    PointStruct(
        id=i,
        vector=[round(random.uniform(-1, 1), 3) for _ in range(4)],
        payload={"label": f"doc-{i}", "group": "A" if i % 3 == 0 else "B"},
    )
    for i in range(50)
]
qd.upsert(collection_name=SCRATCH, points=points)
print(f"Upserted {len(points)} points")

query_vec = [0.1, 0.5, -0.3, 0.8]

# ANN search — uses HNSW graph (approximate, fast)
ann = qd.query_points(
    collection_name=SCRATCH,
    query=query_vec,
    limit=5,
    search_params=SearchParams(exact=False),
).points

# Exact search — brute force, ground truth, ignores HNSW
exact = qd.query_points(
    collection_name=SCRATCH,
    query=query_vec,
    limit=5,
    search_params=SearchParams(exact=True),
).points

ann_ids   = [r.id for r in ann]
exact_ids = [r.id for r in exact]
overlap   = len(set(ann_ids) & set(exact_ids))

print(f"\nANN (HNSW) top-5 IDs  : {ann_ids}")
print(f"Exact (brute) top-5 IDs: {exact_ids}")
print(f"Recall@5               : {overlap}/5 = {overlap/5:.0%}")
print()
print("Note: with 50 points (below full_scan_threshold=10000)")
print("Qdrant uses brute force regardless of exact= flag.")
print("Recall is 100%. HNSW trades recall for speed only at larger scale.")

---
## Part 4 — Payload Indexes and Filtering

By default, filtering on a payload field (`tenant="vault-team"`) does a
**full scan** of every point's payload — O(N). At 100k points with 10 tenants,
every query reads 100k records to find 10k relevant ones.

A **payload index** solves this:

| Index type | Best for |
|------------|----------|
| `KEYWORD` | Exact string match — tenant, doc_type, tag |
| `INTEGER` | Range queries — timestamps, scores |
| `FLOAT`   | Numeric ranges |
| `TEXT`    | Tokenized full-text search within payload |
| `BOOL`    | Boolean flags |

With a keyword index, a tenant filter becomes an inverted-index lookup:
O(1) to find the candidate list, then HNSW only on those candidates.
**For multi-tenant RAG, payload indexes on `tenant` are not optional.**

**→ Qdrant UI:** Collection → **Payload** tab.  
All indexed fields are listed here. The UI also lets you create indexes
interactively — try adding one and watch the API reflect it immediately.

In [ ]:
# Create a keyword index on 'group' in our scratch collection
qd.create_payload_index(
    collection_name=SCRATCH,
    field_name="group",
    field_schema=PayloadSchemaType.KEYWORD,
)
print("Created keyword payload index on 'group'")
print("→ Qdrant UI: Payload tab — 'group' now appears as an indexed field")
print()

# Filtered query — only return points where group=A
results = qd.query_points(
    collection_name=SCRATCH,
    query=query_vec,
    query_filter=Filter(must=[
        FieldCondition(key="group", match=MatchValue(value="A"))
    ]),
    limit=5,
    with_payload=True,
).points

print(f"Filtered query (group=A) returned {len(results)} results:")
for r in results:
    print(f"  id={r.id:3d}  group={r.payload['group']}  score={r.score:.4f}")

# All results must be group=A — assert proves the filter is enforced
assert all(r.payload["group"] == "A" for r in results)
print()
print("✓ All results have group=A — filter is correctly enforced")
print()
print("Without the payload index, this would scan all 50 payloads.")
print("At 1M points, the difference is milliseconds vs seconds.")

---
## Part 5 — Multi-Tenancy via Payload Filtering

Two main patterns for tenant isolation:

**Pattern A — One collection, tenant in payload** (what this demo uses)
```
platform-docs
  Point { tenant: "vault-team",  text: "...", vector: [...] }
  Point { tenant: "ocp-team",    text: "...", vector: [...] }
```
Pros: simple ops, one HNSW index, one collection to monitor.  
Cons: storage isolation is app-level only; a bug in the tenant filter leaks data.

**Pattern B — One collection per tenant**
```
platform-docs-vault-team
platform-docs-ocp-team
```
Pros: full storage isolation, independent HNSW configs per tenant, deleting a
tenant = one `delete_collection()` call.  
Cons: N collections to manage, N HNSW indexes in RAM.

**→ Qdrant UI:** Collection → **Points** tab → use the filter input at the top.
Filter by `tenant = "vault-team"`. You will see exactly the same subset
your application query returns. This is how you verify tenant isolation visually.

In [ ]:
# Re-populate scratch with two tenants
qd.delete_collection(SCRATCH)
qd.create_collection(
    collection_name=SCRATCH,
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
)
# Index before inserting — cheaper than indexing after
qd.create_payload_index(SCRATCH, "tenant",   PayloadSchemaType.KEYWORD)
qd.create_payload_index(SCRATCH, "doc_type", PayloadSchemaType.KEYWORD)

random.seed(0)
tenant_points = []
for i in range(40):
    tenant = "vault-team" if i < 20 else "ocp-team"
    tenant_points.append(PointStruct(
        id=i,
        vector=[round(random.uniform(-1, 1), 3) for _ in range(4)],
        payload={
            "text":     f"Document {i} for {tenant}",
            "tenant":   tenant,
            "doc_type": "runbook" if i % 2 == 0 else "policy",
        },
    ))
qd.upsert(SCRATCH, tenant_points)
print("Inserted 40 points: 20 for vault-team, 20 for ocp-team")
print()

# Query AS vault-team — must never see ocp-team results
vault_q = qd.query_points(
    collection_name=SCRATCH,
    query=[0.1, 0.5, -0.3, 0.8],
    query_filter=Filter(must=[FieldCondition(key="tenant", match=MatchValue(value="vault-team"))]),
    limit=10,
    with_payload=True,
).points

ocp_q = qd.query_points(
    collection_name=SCRATCH,
    query=[0.1, 0.5, -0.3, 0.8],
    query_filter=Filter(must=[FieldCondition(key="tenant", match=MatchValue(value="ocp-team"))]),
    limit=10,
    with_payload=True,
).points

print(f"vault-team query  : {len(vault_q)} results")
print(f"  tenants seen    : {set(r.payload['tenant'] for r in vault_q)}")

print(f"\nocp-team query    : {len(ocp_q)} results")
print(f"  tenants seen    : {set(r.payload['tenant'] for r in ocp_q)}")

assert all(r.payload["tenant"] == "vault-team" for r in vault_q)
assert all(r.payload["tenant"] == "ocp-team"   for r in ocp_q)
print()
print("✓ No cross-tenant leakage")
print()
print("→ Qdrant UI: Points tab → filter by tenant='vault-team'")
print("  You'll see the same 20 points your query returned.")

---
## Part 6 — Quantization: RAM vs Recall

Qdrant stores vectors as **float32** by default (4 bytes per dimension).
Quantization compresses them:

| Type | Bytes/dim | Compression | Recall loss |
|------|-----------|-------------|-------------|
| float32 (none) | 4.0 | 1× | 0% |
| Scalar INT8 | 1.0 | **4×** | <1–2% with rescore |
| Binary | 0.125 | **32×** | 5–15% (use rescore) |

**Rescoring:** after quantized ANN search finds top candidates, Qdrant
re-scores them with the original float32 vectors. This largely recovers
the recall loss at low extra cost.

**Memory at 1M points, 768 dims:**
- float32: `1M × 768 × 4B = 3.07 GB`
- scalar:  `1M × 768 × 1B = 0.77 GB` ← **4× cheaper, <2% recall loss**

Qdrant Cloud pricing is based on RAM. Scalar quantization is often the
single highest-leverage cost optimization in production.

**→ Qdrant UI:** Collection → **Info** tab → Quantization Configuration.  
After creating the collection below, this section shows `Type: Scalar`
and `Always RAM: true`. Without quantization, it shows `None`.

In [ ]:
QUANT_COL = "notebook-quant"

if qd.collection_exists(QUANT_COL):
    qd.delete_collection(QUANT_COL)

qd.create_collection(
    collection_name=QUANT_COL,
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
    quantization_config=ScalarQuantization(
        scalar=ScalarQuantizationConfig(
            type=ScalarType.INT8,
            always_ram=True,   # keep quantized vectors in RAM for fast search
        )
    ),
)
print(f"Created '{QUANT_COL}' with INT8 scalar quantization")

# Insert points and verify quantization is reported by the API
qd.upsert(QUANT_COL, [
    PointStruct(id=i, vector=[round(random.uniform(-1,1),3) for _ in range(4)],
                payload={"label": f"doc-{i}"})
    for i in range(50)
])

info = qd.get_collection(QUANT_COL)
print(f"Quantization config : {info.config.quantization_config}")
print()
print("Memory comparison (768-dim, 100k points, m=16):")
print("  float32  : 100k × 768 × 4B =  295 MB  (vectors) + 25 MB (graph) =  320 MB")
print("  scalar8  : 100k × 768 × 1B =   74 MB  (vectors) + 25 MB (graph) =   99 MB")
print("  binary   : 100k × 768 / 8  =    9 MB  (vectors) + 25 MB (graph) =   34 MB")
print()
print("→ Qdrant UI: Info tab → Quantization Configuration")
print("  Type: Scalar    Always RAM: true")

---
## Part 7 — Real Semantic Query Against Your Corpus

This section uses a stored vector from `platform-docs` as the query.
No Ollama required — we re-query a point against itself to verify the
ANN index is working correctly.

**What scores mean:**
- Cosine score 1.0 = identical vectors (same chunk)
- Score > 0.8 = very semantically similar
- Score 0.5–0.8 = related topic
- Score < 0.5 = weak semantic match

If retrieval quality is poor in production (LLM answers are wrong), the
first thing to check is the score distribution of your queries. If top
scores are below 0.5, your chunks don't semantically match the questions.

**→ Qdrant UI:** Collection → **Search** tab.  
Paste any vector from the Points tab into the search box and run a search.
You'll see results with scores visualized as bars. The **Score threshold**
slider lets you exclude low-confidence results — exactly what the
`score_threshold` parameter does in the API.

In [ ]:
if not qd.collection_exists(COLLECTION):
    print(f"Run 'task w0:migrate' first to populate '{COLLECTION}'")
else:
    # Get one stored point with its vector
    seed_pts, _ = qd.scroll(
        collection_name=COLLECTION,
        limit=1,
        with_payload=True,
        with_vectors=True,
    )

    if not seed_pts:
        print("Collection is empty.")
    else:
        seed = seed_pts[0]
        print(f"Seed point:")
        print(f"  id       : {seed.id}")
        print(f"  source   : {seed.payload.get('source', '?')}")
        print(f"  tenant   : {seed.payload.get('tenant', '?')}")
        print(f"  doc_type : {seed.payload.get('doc_type', '?')}")
        print(f"  dims     : {len(seed.vector)}")
        print(f"  text     : {seed.payload.get('text','')[:120]}...")
        print()

        results = qd.query_points(
            collection_name=COLLECTION,
            query=seed.vector,
            limit=5,
            with_payload=True,
        ).points

        print("Top-5 results for this vector:")
        for i, r in enumerate(results):
            self_flag = "  ← SELF (score should be 1.0)" if r.id == seed.id else ""
            print(f"  #{i+1}  score={r.score:.4f}  id={r.id}{self_flag}")
            print(f"       source   : {r.payload.get('source','?')}")
            print(f"       doc_type : {r.payload.get('doc_type','?')}")
            print(f"       text     : {r.payload.get('text','')[:80]}...")
            print()

        if results[0].id == seed.id:
            print("✓ Self-retrieval confirmed — HNSW index is working correctly")
        else:
            print("⚠ Self-retrieval failed — indexing may not be complete")
            info = qd.get_collection(COLLECTION)
            print(f"  indexed_vectors={info.indexed_vectors_count} / points={info.points_count}")
            print("  Wait a minute and re-run this cell")

---
## Part 8 — Semantic Query With Ollama Embeddings

This section embeds a real question and retrieves relevant chunks.
Requires Ollama running locally with `nomic-embed-text` pulled.

```bash
ollama serve          # start Ollama
ollama pull nomic-embed-text
```

In [ ]:
import ollama

OLLAMA_URL  = os.environ.get("OLLAMA_URL", "http://localhost:11434")
EMBED_MODEL = os.environ.get("EMBED_MODEL", "nomic-embed-text")

ol = ollama.Client(host=OLLAMA_URL)

# Test questions — try changing these
questions = [
    "What is the SPIFFE ID format used in this platform?",
    "How does Vault issue certificates for Consul Connect?",
    "What is the default TTL for leaf certificates?",
]

for question in questions:
    print(f"Question: {question}")
    print("-" * 60)

    # Embed the question
    vec = ol.embed(model=EMBED_MODEL, input=question).embeddings[0]

    # Retrieve top-5 from Qdrant Cloud
    results = qd.query_points(
        collection_name=COLLECTION,
        query=vec,
        limit=5,
        with_payload=True,
    ).points

    for r in results:
        score_bar = "█" * int(r.score * 20)
        print(f"  {score_bar:20s} {r.score:.3f}  {r.payload.get('source','?')}")
        print(f"  {r.payload.get('text','')[:100]}...")
        print()

    print()

---
## Cleanup — delete scratch collections

In [ ]:
for col in [SCRATCH, QUANT_COL]:
    if qd.collection_exists(col):
        qd.delete_collection(col)
        print(f"Deleted '{col}'")

print()
print("Remaining collections:")
for c in qd.get_collections().collections:
    info = qd.get_collection(c.name)
    print(f"  {c.name:30s} {info.points_count:,} points")

---
## Summary: Qdrant UI cheat sheet

| What to check | Where in the UI |
|---|---|
| Collection list + point counts | Cluster → Collections tab |
| HNSW m, ef_construct, distance | Collection → Info tab |
| Quantization type | Collection → Info tab |
| Optimizer status, segment count | Collection → Info tab |
| Indexed payload fields | Collection → Payload tab |
| Browse points with filters | Collection → Points tab → filter input |
| Run a test search | Collection → Search tab |
| Cluster RAM / CPU usage | Cluster → Metrics tab |

**Debugging retrieval quality:**
1. Points tab — are payloads correct? tenant and doc_type set?
2. Info tab — is `indexed_vectors == points_count`? If not, optimizer is still running.
3. Search tab — paste a real query vector, check score distribution.
4. If top scores < 0.5: chunking or embedding model problem, not Qdrant.
5. If scores are good but LLM answers wrong: chunks are topically similar but
   informationally wrong — chunks are too large or overlap too aggressively.